# Inventario de shortcuts de Microsoft Fabric

Este notebook escanea workspaces de Microsoft Fabric y construye un **inventario completo de los shortcuts de OneLake**, con validaciones de calidad y governance:

- **Descubre** workspaces e ítems (Lakehouses, KQL Databases, Mirrored Databases) mediante la [API REST de Fabric](https://learn.microsoft.com/rest/api/fabric/).
- **Extrae** todos los shortcuts de cada ítem y resuelve su destino (OneLake interno o almacenamiento externo: S3, ADLS Gen2, GCS, Dataverse, etc.).
- **Valida**: detecta shortcuts **huérfanos** (el destino ya no existe en el ámbito escaneado), **circulares** (cadenas de shortcuts que forman un ciclo) y **externos sin governance** (sin etiqueta de sensibilidad ni endorsement).
- **Presenta** un informe HTML interactivo con vistas filtradas y, opcionalmente, **persiste** el resultado en una tabla Delta para análisis histórico.

**Requisitos**: ejecutarse dentro de un notebook de Fabric (usa `notebookutils`, `sempy` y `displayHTML`). El usuario o service principal necesita, como mínimo, rol de *Viewer* en los workspaces a escanear. Para `SCOPE_MODE = "all"` con cobertura de tenant completo y para el enriquecimiento de governance se necesitan permisos de administrador de Fabric.

El flujo es secuencial: ejecuta las celdas en orden. Toda la configuración se hace en la celda de parámetros siguiente.

## Parámetros

Única celda que necesitas modificar. Referencia completa:

| Parámetro | Valores | Descripción |
|---|---|---|
| `SCOPE_MODE` | `"all"` \| `"list"` | `"all"` escanea todos los workspaces del tenant (vía API de admin si hay permisos; si no, los workspaces accesibles por el usuario). `"list"` limita el escaneo a los workspaces de `WORKSPACE_LIST`. |
| `WORKSPACE_LIST` | lista de `str` | Nombres o IDs (GUID) de los workspaces a escanear. Solo se usa con `SCOPE_MODE = "list"`. |
| `RESOLVE_NAMES` | `True` \| `False` | Si `True`, las entradas de `WORKSPACE_LIST` pueden ser nombres visibles además de IDs; el notebook resuelve el ID automáticamente. |
| `AUTH_MODE` | `"user"` \| `"sp"` | `"user"`: token delegado del usuario que ejecuta el notebook (on-behalf-of vía `notebookutils`). `"sp"`: service principal, recomendado para ejecuciones programadas o escaneo de tenant completo. |
| `SP_TENANT_ID` | GUID | ID del tenant de Entra ID. Solo con `AUTH_MODE = "sp"`. |
| `SP_CLIENT_ID` | GUID | Client ID de la app registration del service principal. Solo con `AUTH_MODE = "sp"`. |
| `SP_KEYVAULT` | URL | URL del Azure Key Vault donde está guardado el secreto del service principal (nunca se escribe el secreto en el notebook). |
| `SP_SECRET_NAME` | `str` | Nombre del secreto dentro del Key Vault. |
| `GOVERNANCE_CHECK` | `True` \| `False` | Si `True`, enriquece cada ítem con su etiqueta de sensibilidad y endorsement (vía admin scan de `sempy`) y marca los shortcuts externos sin governance. Requiere permisos de admin; si no los hay, degrada sin fallar. |
| `SAVE_TO_DELTA` | `True` \| `False` | Si `True`, guarda el inventario final en una tabla Delta del Lakehouse adjunto (modo *overwrite*). |
| `DELTA_TABLE` | `str` | Nombre de la tabla Delta destino. Solo se usa con `SAVE_TO_DELTA = True`. |
| `MAX_WORKERS` | `int` | Número de hilos en paralelo para las llamadas a la API de shortcuts. Súbelo para escaneos grandes; bájalo si aparecen throttling (429) frecuentes. |

In [ ]:
# === Parameters ===
SCOPE_MODE       = "list"          # "all" | "list"
WORKSPACE_LIST   = ["Mi Workspace"]  # Only used if SCOPE_MODE = "list"
RESOLVE_NAMES    = True
AUTH_MODE        = "user"          # "user" | "sp"
SP_TENANT_ID     = ""
SP_CLIENT_ID     = ""
SP_KEYVAULT      = ""
SP_SECRET_NAME   = ""
GOVERNANCE_CHECK = True
SAVE_TO_DELTA    = False
DELTA_TABLE      = "shortcut_inventory"
MAX_WORKERS      = 8

In [ ]:
import datetime as _dt

def log(msg: str) -> None:
    print(f"[{_dt.datetime.now(_dt.timezone.utc).isoformat(timespec='seconds')}] {msg}")

log("Parameters loaded.")

## Autenticación

Construye el cliente HTTP que usará todo el notebook según `AUTH_MODE`:

- **`_UserClient`** (`AUTH_MODE = "user"`): obtiene un token delegado del usuario actual con `notebookutils.credentials.getToken()`. No requiere configuración adicional, pero solo ve los workspaces a los que el usuario tiene acceso (`is_admin = False`).
- **`_SpClient`** (`AUTH_MODE = "sp"`): autentica un service principal con `azure-identity`. El secreto se lee de Azure Key Vault en tiempo de ejecución — nunca se almacena en el notebook. Se asume que tiene permisos de admin (`is_admin = True`); si no los tiene, las llamadas de admin degradan sin romper el flujo.

Ambos clientes exponen el mismo método `get(url)`, que renueva el token en cada llamada, por lo que el resto del notebook es agnóstico al modo de autenticación.

In [ ]:
import sempy.fabric as fabric

class _UserClient:
    """User token via notebookutils OBO — includes delegated Fabric API scopes."""
    def __init__(self):
        import requests as _req
        self._session = _req.Session()
        self.is_admin = False
    def _token(self):
        return notebookutils.credentials.getToken("https://api.fabric.microsoft.com")
    def get(self, url):
        if url.startswith("/"):
            url = "https://api.fabric.microsoft.com" + url
        self._session.headers["Authorization"] = f"Bearer {self._token()}"
        resp = self._session.get(url)
        resp.raise_for_status()
        return resp

class _SpClient:
    """Service principal client using azure-identity + requests."""
    def __init__(self, tenant_id, client_id, keyvault, secret_name):
        import requests
        from azure.identity import ClientSecretCredential
        secret = notebookutils.credentials.getSecret(keyvault, secret_name)
        self._cred = ClientSecretCredential(tenant_id, client_id, secret)
        self._session = requests.Session()
        self.is_admin = True  # assumed; admin calls still degrade gracefully if not
    def _token(self):
        return self._cred.get_token("https://api.fabric.microsoft.com/.default").token
    def get(self, url):
        if url.startswith("/"):
            url = "https://api.fabric.microsoft.com" + url
        self._session.headers["Authorization"] = f"Bearer {self._token()}"
        return self._session.get(url)

def build_client(auth_mode, sp_tenant_id, sp_client_id, sp_keyvault, sp_secret_name):
    if auth_mode == "sp":
        return _SpClient(sp_tenant_id, sp_client_id, sp_keyvault, sp_secret_name)
    return _UserClient()

client = build_client(AUTH_MODE, SP_TENANT_ID, SP_CLIENT_ID, SP_KEYVAULT, SP_SECRET_NAME)
log(f"Client built: auth_mode={AUTH_MODE}, is_admin={client.is_admin}")

## Descubrimiento de workspaces

Resuelve la lista de workspaces a escanear según `SCOPE_MODE`:

- Con `"all"`: intenta primero la API de administración (`/v1/admin/workspaces`, cobertura de todo el tenant) y, si el cliente no es admin, cae a `/v1/workspaces` (solo los workspaces accesibles).
- Con `"list"`: acepta IDs o, si `RESOLVE_NAMES = True`, nombres visibles, y los valida contra los workspaces accesibles. Las entradas no encontradas se registran en el log y se omiten.

El helper `get_paged()` centraliza la paginación (`continuationToken`) y los reintentos ante throttling **HTTP 429** con backoff exponencial, respetando la cabecera `Retry-After`. Todas las llamadas a la API del notebook pasan por él.

In [ ]:
import time

def get_paged(client, url, value_key="value", max_retries=5):
    items, next_url = [], url
    while next_url:
        for attempt in range(max_retries):
            resp = client.get(next_url)
            if resp.status_code == 429:
                wait = int(resp.headers.get("Retry-After", 2 ** attempt))
                log(f"429 on {next_url}; retrying in {wait}s")
                time.sleep(wait)
                continue
            break
        if resp.status_code < 200 or resp.status_code >= 300:
            log(f"GET {next_url} -> {resp.status_code}: {resp.text[:200]}")
            return items
        body = resp.json()
        items.extend(body.get(value_key, []))
        token = body.get("continuationToken")
        next_url = body.get("continuationUri") if token else None
    return items

In [ ]:
def resolve_workspaces(client, scope_mode, workspace_list, resolve_names):
    if scope_mode == "all":
        admin = get_paged(client, "/v1/admin/workspaces") if client.is_admin else []
        if admin:
            return [{"id": w["id"], "name": w.get("name") or w.get("displayName")} for w in admin]
        ws = get_paged(client, "/v1/workspaces")
        return [{"id": w["id"], "name": w.get("displayName")} for w in ws]
    # scope_mode == "list"
    all_ws = get_paged(client, "/v1/workspaces")
    by_id = {w["id"]: w.get("displayName") for w in all_ws}
    by_name = {w.get("displayName"): w["id"] for w in all_ws}
    out = []
    for entry in workspace_list:
        if entry in by_id:
            out.append({"id": entry, "name": by_id[entry]})
        elif resolve_names and entry in by_name:
            out.append({"id": by_name[entry], "name": entry})
        else:
            log(f"Workspace not found/accessible: {entry}")
    return out

workspaces = resolve_workspaces(client, SCOPE_MODE, WORKSPACE_LIST, RESOLVE_NAMES)
log(f"Resolved {len(workspaces)} workspace(s): {[w['name'] for w in workspaces]}")

## Descubrimiento de ítems

Lista todos los ítems de cada workspace (`/v1/workspaces/{id}/items`) y los aplana en filas con workspace, ID, nombre y tipo de ítem. Este catálogo se usa después con dos fines: saber a qué ítems pedirles sus shortcuts y validar si los destinos OneLake existen dentro del ámbito escaneado (detección de huérfanos).

In [ ]:
def discover_items(client, workspaces):
    out = []
    for ws in workspaces:
        raw = get_paged(client, f"/v1/workspaces/{ws['id']}/items")
        for it in raw:
            out.append({
                "workspace_id": ws["id"],
                "workspace_name": ws["name"],
                "item_id": it["id"],
                "item_name": it.get("displayName"),
                "item_type": it.get("type"),
            })
    return out

items = discover_items(client, workspaces)
log(f"Discovered {len(items)} item(s) across {len(workspaces)} workspace(s)")

## Parseo de destinos de shortcuts

Normaliza el objeto `target` que devuelve la API para cada shortcut a un esquema plano común. Distingue dos casos:

- **OneLake** (interno): extrae workspace, ítem y subruta del destino, y construye una URI `onelake://workspace/item/ruta`. Estos son los únicos shortcuts que participan en la detección de huérfanos y ciclos.
- **Externos** (Amazon S3, ADLS Gen2, Google Cloud Storage, S3 compatible, Dataverse, Azure Blob Storage, OneDrive/SharePoint): concatena `location + subpath` y los marca con `is_external = True`, lo que los hace candidatos a la revisión de governance.

In [ ]:
_EXTERNAL_TYPES = {"AmazonS3", "AdlsGen2", "GoogleCloudStorage",
                   "S3Compatible", "Dataverse", "AzureBlobStorage",
                   "OneDriveSharePoint"}
# maps target_type -> key in the target dict holding its details
_DETAIL_KEY = {
    "OneLake": "oneLake", "AmazonS3": "amazonS3", "AdlsGen2": "adlsGen2",
    "GoogleCloudStorage": "googleCloudStorage", "S3Compatible": "s3Compatible",
    "Dataverse": "dataverse", "AzureBlobStorage": "azureBlobStorage",
    "OneDriveSharePoint": "oneDriveSharePoint",
}

def parse_shortcut_target(target):
    ttype = target.get("type")
    detail = target.get(_DETAIL_KEY.get(ttype, ""), {}) or {}
    if ttype == "OneLake":
        path = detail.get("path", "")
        wsid = detail.get("workspaceId")
        itid = detail.get("itemId")
        return {
            "target_type": "OneLake",
            "target_workspace_id": wsid,
            "target_item_id": itid,
            "target_subpath": path,
            "target_location": f"onelake://{wsid}/{itid}/{path}",
            "is_external": False,
        }
    location = detail.get("location", "")
    subpath = detail.get("subpath", "") or ""
    return {
        "target_type": ttype,
        "target_workspace_id": None,
        "target_item_id": None,
        "target_subpath": subpath,
        "target_location": f"{location}{subpath}",
        "is_external": ttype in _EXTERNAL_TYPES,
    }


## Extracción de shortcuts

Llama a `/v1/workspaces/{ws}/items/{item}/shortcuts` para cada ítem que soporta shortcuts (solo **Lakehouse**, **KQLDatabase** y **MirroredDatabase** exponen este endpoint; el resto se omite y se registra en el log). Las llamadas se paralelizan con un `ThreadPoolExecutor` de `MAX_WORKERS` hilos para acelerar escaneos grandes. Si un ítem falla, se conserva una fila con la columna `error` en lugar de abortar el escaneo completo.

In [ ]:
def _base_row(item):
    return {
        "workspace_id": item["workspace_id"],
        "workspace_name": item["workspace_name"],
        "item_id": item["item_id"],
        "item_name": item["item_name"],
        "item_type": item["item_type"],
        "shortcut_name": None, "shortcut_path": None,
        "target_type": None, "target_workspace_id": None,
        "target_item_id": None, "target_subpath": None,
        "target_location": None, "is_external": None,
    }

def extract_item_shortcuts(client, item):
    url = f"/v1/workspaces/{item['workspace_id']}/items/{item['item_id']}/shortcuts"
    try:
        scs = get_paged(client, url)
    except Exception as e:
        return [{**_base_row(item), "error": f"{type(e).__name__}: {e}"}]
    out = []
    for sc in scs:
        parsed = parse_shortcut_target(sc.get("target", {}))
        out.append({
            **_base_row(item),
            "shortcut_name": sc.get("name"),
            "shortcut_path": sc.get("path"),
            **parsed,
            "error": None,
        })
    return out

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import functools

# ponytail: only Fabric item types that expose the /shortcuts REST endpoint
_SHORTCUT_SUPPORTED_TYPES = {"Lakehouse", "KQLDatabase", "MirroredDatabase"}

def extract_all(client, items, max_workers):
    supported = [it for it in items if it["item_type"] in _SHORTCUT_SUPPORTED_TYPES]
    skipped = len(items) - len(supported)
    if skipped:
        log(f"Skipping {skipped} item(s) — type not in {_SHORTCUT_SUPPORTED_TYPES}")
    rows = []
    _extract = functools.partial(extract_item_shortcuts, client)
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        for res in ex.map(_extract, supported):
            rows.extend(res)
    return [r for r in rows if r["shortcut_name"] is not None or r["error"]]

rows = extract_all(client, items, MAX_WORKERS)
log(f"Extracted {len([r for r in rows if r['shortcut_name']])} shortcut(s); "
    f"{len([r for r in rows if r['error']])} item error(s)")


## Validación: shortcuts huérfanos

Marca como **huérfano** (`is_orphan = True`) todo shortcut OneLake cuyo destino (workspace + ítem) no existe entre los ítems descubiertos, con el motivo en `orphan_reason`. Suele indicar que el ítem de destino fue borrado y el shortcut quedó roto.

> **Nota**: la comprobación se hace contra el ámbito escaneado. Con `SCOPE_MODE = "list"`, un destino que vive en un workspace fuera de la lista aparecerá como huérfano aunque exista — amplía el ámbito para confirmarlo.

In [ ]:
def flag_orphans(rows, items):
    known = {(it["workspace_id"], it["item_id"]) for it in items}
    for r in rows:
        if r.get("target_type") == "OneLake":
            key = (r.get("target_workspace_id"), r.get("target_item_id"))
            if key in known:
                r["is_orphan"], r["orphan_reason"] = False, None
            else:
                r["is_orphan"] = True
                r["orphan_reason"] = (
                    f"OneLake target item {key[1]} in workspace {key[0]} "
                    "not found in scanned scope")
        else:
            r["is_orphan"], r["orphan_reason"] = False, None
    return rows


## Validación: shortcuts circulares

Construye un grafo dirigido donde cada shortcut OneLake es una arista *ítem origen → ítem destino* y busca ciclos mediante DFS con coloreado (una arista de retroceso hacia un nodo en curso indica ciclo). Los shortcuts cuya arista participa en un ciclo se marcan con `is_circular = True` y la arista concreta en `circular_path`. Un ciclo (A apunta a B y B apunta a A, directa o transitivamente) puede causar resoluciones infinitas o datos duplicados y conviene deshacerlo.

In [ ]:
def detect_cycles(rows):
    # default
    for r in rows:
        r.setdefault("is_circular", False)
        r.setdefault("circular_path", None)
    # build adjacency from OneLake edges
    edges = {}  # src_node -> set(dst_node)
    def node(ws, it): return f"{ws}/{it}"
    for r in rows:
        if r.get("target_type") == "OneLake" and r.get("target_item_id"):
            s = node(r["workspace_id"], r["item_id"])
            d = node(r["target_workspace_id"], r["target_item_id"])
            edges.setdefault(s, set()).add(d)
    # find nodes that participate in any cycle (Tarjan-lite via DFS)
    WHITE, GRAY, BLACK = 0, 1, 2
    color, cyclic_edges = {}, set()
    def dfs(u, path):
        color[u] = GRAY
        path.append(u)
        for v in edges.get(u, ()):  # noqa
            if color.get(v, WHITE) == GRAY:        # back-edge -> cycle
                i = path.index(v)
                for a, b in zip(path[i:], path[i+1:] + [v]):
                    cyclic_edges.add((a, b))
            elif color.get(v, WHITE) == WHITE:
                dfs(v, path)
        path.pop()
        color[u] = BLACK
    for n in list(edges):
        if color.get(n, WHITE) == WHITE:
            dfs(n, [])
    # mark rows whose edge is part of a cycle
    for r in rows:
        if r.get("target_type") == "OneLake" and r.get("target_item_id"):
            s = node(r["workspace_id"], r["item_id"])
            d = node(r["target_workspace_id"], r["target_item_id"])
            if (s, d) in cyclic_edges:
                r["is_circular"] = True
                r["circular_path"] = f"{s} -> {d}"
    return rows


## Enriquecimiento de governance

Solo se ejecuta si `GOVERNANCE_CHECK = True`. Obtiene metadatos de governance de los ítems mediante el *admin scan* de `sempy` (`fabric.admin.scan_workspaces`, requiere permisos de administrador de Fabric) y añade a cada fila:

- `item_sensitivity_label`: etiqueta de sensibilidad del ítem que contiene el shortcut.
- `item_endorsement`: endorsement del ítem (`Certified` o `Promoted`).
- `governance_flag = True` cuando un shortcut **externo** vive en un ítem sin etiqueta de sensibilidad ni endorsement — es decir, datos que salen o entran de fuentes externas sin ningún control de governance declarado.

Si el scan no está disponible (falta de permisos o de API), se registra en el log y el notebook continúa sin enriquecer.

In [ ]:
_GOOD_ENDORSEMENTS = {"Certified", "Promoted"}

def apply_governance(rows, gov):
    for r in rows:
        meta = gov.get(r.get("item_id"), {})
        sens = meta.get("sensitivity")
        endo = meta.get("endorsement")
        r["item_sensitivity_label"] = sens
        r["item_endorsement"] = endo
        if r.get("is_external"):
            r["governance_flag"] = not sens and endo not in _GOOD_ENDORSEMENTS
        else:
            r["governance_flag"] = False
    return rows


In [ ]:
def fetch_governance(workspaces):
    gov = {}
    try:
        import sempy.fabric as fabric
        ids = [w["id"] for w in workspaces][:100]
        df = fabric.admin.scan_workspaces(workspace=ids, return_dataframe=True)
    except Exception as e:
        log(f"Governance scan unavailable: {type(e).__name__}: {e}")
        return gov
    # df has one row per workspace; each contains item collections. Flatten lakehouse-like items.
    for _, ws_row in df.iterrows():
        for coll in ("Lakehouses", "Datasets", "Items"):
            for it in (ws_row.get(coll) or []):
                iid = it.get("id") or it.get("objectId")
                if iid:
                    gov[iid] = {
                        "sensitivity": (it.get("sensitivityLabel") or {}).get("labelId")
                                        if isinstance(it.get("sensitivityLabel"), dict)
                                        else it.get("sensitivityLabel"),
                        "endorsement": (it.get("endorsementDetails") or {}).get("endorsement")
                                        if isinstance(it.get("endorsementDetails"), dict)
                                        else it.get("endorsement"),
                    }
    return gov

gov = fetch_governance(workspaces) if GOVERNANCE_CHECK else {}
rows = apply_governance(rows, gov)
log(f"Governance: {len(gov)} item(s) enriched; "
    f"{len([r for r in rows if r.get('governance_flag')])} flagged")


## Resolución de nombres y finalización

Convierte los GUIDs de los destinos OneLake en nombres legibles (`target_workspace_name`, `target_item_name`) y construye `target_location_display`, una URI legible tipo `onelake://Ventas/LakehouseVentas/Tables/clientes`. Primero busca en el catálogo ya descubierto; si el destino está fuera del ámbito escaneado, hace una consulta puntual a la API (con caché para no repetir llamadas).

Después, `finalize()` sella cada fila con `scan_timestamp` (UTC) y garantiza que todas las filas tengan el mismo esquema de columnas (`_ALL_COLS`), rellenando con `None` lo que falte.

In [ ]:
_ALL_COLS = ["scan_timestamp","workspace_id","workspace_name","item_id","item_name",
             "item_type","shortcut_name","shortcut_path","target_type","target_workspace_id",
             "target_workspace_name","target_item_id","target_item_name","target_subpath",
             "target_location","target_location_display","is_external","is_orphan","orphan_reason",
             "is_circular","circular_path","item_sensitivity_label","item_endorsement",
             "governance_flag","error"]

def resolve_target_names(rows, items, workspaces, client=None):
    """Resolve OneLake target workspace/item ids to display names.

    Falls back to a live lookup via `client` when the target is outside the
    scanned scope (e.g. a shortcut pointing to a workspace not in
    WORKSPACE_LIST), so cross-workspace targets still get readable names.
    """
    item_name = {(it["workspace_id"], it["item_id"]): it["item_name"] for it in items}
    ws_name = {w["id"]: w["name"] for w in workspaces}
    _ws_cache, _item_cache = {}, {}

    def _fetch_ws_name(wsid):
        if not wsid or client is None:
            return None
        if wsid not in _ws_cache:
            try:
                _ws_cache[wsid] = client.get(f"/v1/workspaces/{wsid}").json().get("displayName")
            except Exception as e:
                log(f"Could not resolve workspace name for {wsid}: {type(e).__name__}: {e}")
                _ws_cache[wsid] = None
        return _ws_cache[wsid]

    def _fetch_item_name(wsid, itid):
        if not wsid or not itid or client is None:
            return None
        key = (wsid, itid)
        if key not in _item_cache:
            try:
                _item_cache[key] = client.get(
                    f"/v1/workspaces/{wsid}/items/{itid}").json().get("displayName")
            except Exception as e:
                log(f"Could not resolve item name for {itid} in {wsid}: {type(e).__name__}: {e}")
                _item_cache[key] = None
        return _item_cache[key]

    for r in rows:
        if r.get("target_type") == "OneLake":
            wsid, itid = r.get("target_workspace_id"), r.get("target_item_id")
            r["target_workspace_name"] = ws_name.get(wsid) or _fetch_ws_name(wsid)
            r["target_item_name"] = item_name.get((wsid, itid)) or _fetch_item_name(wsid, itid)
            wsn = r["target_workspace_name"] or wsid or "?"
            itn = r["target_item_name"] or itid or "?"
            subpath = r.get("target_subpath") or ""
            r["target_location_display"] = f"onelake://{wsn}/{itn}/{subpath}"
        else:
            r["target_workspace_name"] = None
            r["target_item_name"] = None
            r["target_location_display"] = r.get("target_location")
    return rows

def finalize(rows):
    ts = _dt.datetime.now(_dt.timezone.utc).isoformat(timespec="seconds")
    for r in rows:
        r["scan_timestamp"] = ts
        for c in _ALL_COLS:
            r.setdefault(c, None)
    return rows

final_rows = finalize(resolve_target_names(rows, items, workspaces, client))
log(f"Finalized {len(final_rows)} row(s)")

## Informe HTML

Renderiza el inventario como un informe interactivo dentro del propio notebook (`displayHTML`), sin dependencias externas. Incluye:

- **Resumen**: totales de shortcuts, huérfanos, circulares, flags de governance y desglose por tipo de destino.
- **Cuatro vistas** con pestañas: *Vista General* (todo), *Circulares*, *Orphan* y *Governance*, cada una con su contador.
- **Código de colores por fila**: rojo = huérfano, naranja = circular, amarillo = flag de governance.

Todo el contenido se escapa con `html.escape()` antes de renderizarse.

In [ ]:
import html as _html_mod

def _row_color(r):
    if r.get("is_orphan"):       return "#f8d7da"
    if r.get("is_circular"):     return "#ffe5b4"
    if r.get("governance_flag"): return "#fff3cd"
    return "#ffffff"

_COLS = ["workspace_name","item_name","shortcut_name","shortcut_path",
         "target_type","target_location_display","target_workspace_name","target_item_name",
         "is_orphan","is_circular","governance_flag","item_endorsement","error"]

def _build_table(rows):
    if not rows:
        return "<p style='color:#666;font-style:italic'>No items in this view.</p>"
    head = "".join(
        f"<th style='text-align:left;padding:6px 8px;border-bottom:2px solid #dee2e6;"
        f"background:#f8f9fa;position:sticky;top:0'>{c}</th>" for c in _COLS)
    body = []
    for r in rows:
        cells = "".join(
            f"<td style='padding:5px 8px;border-bottom:1px solid #e9ecef'>"
            f"{_html_mod.escape(str(r.get(c) or ''))}</td>" for c in _COLS)
        body.append(f"<tr style='background:{_row_color(r)}'>{cells}</tr>")
    return (
        "<div style='overflow-x:auto;max-height:500px;overflow-y:auto'>"
        "<table style='border-collapse:collapse;font-family:sans-serif;"
        "font-size:12px;width:100%'>"
        f"<thead><tr>{head}</tr></thead><tbody>{''.join(body)}</tbody></table></div>"
    )

def render_html(rows):
    total      = len(rows)
    n_orphan   = sum(1 for r in rows if r.get("is_orphan"))
    n_circular = sum(1 for r in rows if r.get("is_circular"))
    n_gov      = sum(1 for r in rows if r.get("governance_flag"))
    by_type    = {}
    for r in rows:
        t = r.get("target_type") or "unknown"
        by_type[t] = by_type.get(t, 0) + 1

    views = [
        ("vista-general", "Vista General", rows,                                     "#0078d4"),
        ("circulares",    "Circulares",    [r for r in rows if r.get("is_circular")],  "#d97706"),
        ("orphan",        "Orphan",        [r for r in rows if r.get("is_orphan")],    "#dc2626"),
        ("governance",    "Governance",    [r for r in rows if r.get("governance_flag")], "#7c3aed"),
    ]

    btn_css = (
        "display:inline-flex;align-items:center;gap:6px;padding:8px 16px;"
        "border:none;border-radius:6px;cursor:pointer;font-size:13px;"
        "font-family:sans-serif;font-weight:500;transition:opacity .15s"
    )

    buttons, panels = [], []
    for i, (vid, label, vrows, color) in enumerate(views):
        active = f"background:{color};color:#fff;box-shadow:0 2px 6px {color}66"
        idle   = "background:#f3f4f6;color:#374151"
        cnt    = len(vrows)
        buttons.append(
            f"<button id='btn-{vid}' onclick=\"showView('{vid}')\" "
            f"style='{btn_css};{active if i == 0 else idle}'>"
            f"{label}"
            f"<span style='background:rgba(255,255,255,0.25);border-radius:12px;"
            f"padding:2px 7px;font-size:11px'>{cnt}</span>"
            f"</button>"
        )
        display = 'block' if i == 0 else 'none'
        panels.append(
            f"<div id='panel-{vid}' style='display:{display}'>"
            f"{_build_table(vrows)}</div>"
        )

    summary = (
        "<div style='font-family:sans-serif;font-size:13px;color:#374151;"
        "margin-bottom:14px;padding:10px 14px;background:#f8f9fa;border-radius:6px;"
        "border-left:4px solid #0078d4'>"
        f"<b>Total shortcuts:</b> {total} &nbsp;·&nbsp; "
        f"<b>Orphan:</b> {n_orphan} &nbsp;·&nbsp; "
        f"<b>Circular:</b> {n_circular} &nbsp;·&nbsp; "
        f"<b>Governance flags:</b> {n_gov}<br>"
        "<span style='color:#6b7280'><b>By target type:</b> "
        + ", ".join(f"{k}: {v}" for k, v in sorted(by_type.items()))
        + "</span></div>"
    )

    colors_js = (
        "{"
        + ",".join(f"'{vid}':'{color}'" for vid, _, __, color in views)
        + "}"
    )
    vids_js = str([vid for vid, *_ in views])

    script = (
        "<script>\n"
        f"var _c={colors_js};var _v={vids_js};\n"
        "function showView(id){\n"
        "  _v.forEach(function(v){\n"
        "    document.getElementById('panel-'+v).style.display=v===id?'block':'none';\n"
        "    var b=document.getElementById('btn-'+v);\n"
        "    if(v===id){b.style.background=_c[v];b.style.color='#fff';"
        "b.style.boxShadow='0 2px 6px '+_c[v]+'66';}"
        "else{b.style.background='#f3f4f6';b.style.color='#374151';"
        "b.style.boxShadow='none';}});\n"
        "}\n"
        "</script>"
    )

    btn_bar = (
        "<div style='display:flex;gap:8px;margin-bottom:14px;flex-wrap:wrap'>"
        + "".join(buttons)
        + "</div>"
    )
    legend = (
        "<div style='font-family:sans-serif;font-size:11px;margin-top:10px;display:flex;gap:12px'>"
        "<span style='background:#f8d7da;padding:2px 8px;border-radius:4px'>orphan</span>"
        "<span style='background:#ffe5b4;padding:2px 8px;border-radius:4px'>circular</span>"
        "<span style='background:#fff3cd;padding:2px 8px;border-radius:4px'>governance</span>"
        "</div>"
    )

    return (
        "<div style='padding:16px'>"
        + summary + btn_bar + "".join(panels) + legend + script
        + "</div>"
    )

In [ ]:
displayHTML(render_html(final_rows))

## Opcional: guardado en tabla Delta

Solo se ejecuta si `SAVE_TO_DELTA = True`. Convierte el inventario en un DataFrame de Spark con el esquema fijo de `_ALL_COLS` y lo escribe en la tabla `DELTA_TABLE` del Lakehouse adjunto en modo **overwrite** (cada ejecución reemplaza la anterior; el `scan_timestamp` identifica el escaneo). Útil para consultar el inventario con SQL, montar informes de Power BI encima o, cambiando el modo a `append`, conservar histórico de escaneos.

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, BooleanType

_BOOL_COLS = {"is_external", "is_orphan", "is_circular", "governance_flag"}

_SPARK_SCHEMA = StructType([
    StructField(c, BooleanType() if c in _BOOL_COLS else StringType(), True)
    for c in _ALL_COLS
])

def to_spark_df(rows):
    # explicit schema avoids CANNOT_DETERMINE_TYPE when a column is all-None
    # across every row (type inference has nothing to sample from)
    ordered = [{c: r.get(c) for c in _ALL_COLS} for r in rows]
    return spark.createDataFrame(ordered, schema=_SPARK_SCHEMA) if ordered else None

if SAVE_TO_DELTA:
    sdf = to_spark_df(final_rows)
    if sdf is not None:
        sdf.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(DELTA_TABLE)
        log(f"Saved {sdf.count()} row(s) to Delta table '{DELTA_TABLE}' (overwrite)")
    else:
        log("No rows to save; skipped Delta write.")
else:
    log("SAVE_TO_DELTA is False; skipped Delta write.")
